In [ ]:
import pandas as pd

In [ ]:
def load_data():
    df = pd.read_csv("Medical Diagnosis Expert System.csv")
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    df = df.drop_duplicates()
    df = df.dropna()

    df["disease"] = df["disease"].str.strip().str.lower().str.replace(" ", "_")
    df["symptoms"] = df["symptoms"].apply(
        lambda x: [s.strip().lower().replace(" ", "_") for s in x.split(",")]
    )
    df["precautions"] = df["precautions"].apply(
        lambda x: [p.strip().lower() for p in x.split(",")]
    )

    df = df.reset_index(drop=True)
    df["pattern_id"] = df.index

    patterns = []
    for _, row in df.iterrows():
        patterns.append({
            "pattern_id": row["pattern_id"],
            "disease": row["disease"],
            "symptoms": row["symptoms"],
            "precautions": row["precautions"]
        })

    all_symptoms = sorted(set(
        symptom
        for pattern in patterns
        for symptom in pattern["symptoms"]
    ))

    matrix_data = []
    for pattern in patterns:
        row_vector = [1 if symptom in pattern["symptoms"] else 0
                      for symptom in all_symptoms]
        matrix_data.append(row_vector)

    pattern_matrix = pd.DataFrame(
        matrix_data,
        columns=all_symptoms,
        index=df["pattern_id"]
    )

    disease_index = {}
    for pattern in patterns:
        disease = pattern["disease"]
        pid = pattern["pattern_id"]
        if disease not in disease_index:
            disease_index[disease] = []
        disease_index[disease].append(pid)

    return patterns, pattern_matrix, all_symptoms, disease_index

In [ ]:
patterns, pattern_matrix, all_symptoms, disease_index = load_data()

print("Patterns count:", len(patterns))
print("Unique symptoms:", len(all_symptoms))
print("Matrix shape:", pattern_matrix)
print("Unique diseases:", len(disease_index))

# Pick first pattern and verify matrix matches
p = patterns[0]
print("\nPattern 0 disease:", p["disease"])
print("Pattern 0 symptoms:", p["symptoms"])
print("Matrix row 0 (only 1s):", 
      pattern_matrix.loc[0][pattern_matrix.loc[0] == 1].index.tolist())
# These two lists must match exactly

In [ ]:
all_symptoms

In [ ]:
# Tokenization using NLTK
from functools import reduce
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
tokenizer = RegexpTokenizer(r'\w+')

In [ ]:
stemmed_symptom_dict = {}

for symptom in all_symptoms:
    
    words = symptom.replace("_", " ")
    words = tokenizer.tokenize(words)
    words = [w for w in words if w not in stop_words]
    
    stemmed_words = frozenset(ps.stem(w) for w in words)
    stemmed_symptom_dict[stemmed_words] = symptom

In [ ]:
sent = "i have abdominal pain and acidity and belly pain what to do i feel like why has had of on one two over in "
tokens = tokenizer.tokenize(sent)

filtered_tokens = [word for word in tokens if word not in stop_words]

print("filtered tokens = ",filtered_tokens)
stemmed_s = []
for w in filtered_tokens:
    w.replace("_"," ")
    w = ps.stem(w)
    stemmed_s.append(w)
print("stemmed tokens = ",stemmed_s)

matched = []
for stemmed_key, original_symptom in stemmed_symptom_dict.items():
    if stemmed_key.issubset(stemmed_s):
        matched.append(original_symptom)
        
print("symptom = ",matched)

if len(matched_symptoms) == 0:
        print("\nSorry, I could not recognize any symptoms from what you described.")
        print("Please try to describe your symptoms more clearly.")
        print("Example: 'I have itching, skin rash and fever'")

In [ ]:
stemmed_symptom_dict

In [ ]:
sent = "i have abdominal pain and acidity and belly pain what to do i feel like why has had of on one two over in "
tokens = tokenizer.tokenize(sent)

matched = []
for stemmed_key, original_symptom in stemmed_symptom_dict.items():
    if stemmed_key.issubset(tokens):
        matched.append(original_symptom)
        
print(matched)

In [ ]:
all_symptoms_stemmed = []

for symptom in all_symptoms:
    words = symptom.replace("_"," ")
    words = tokenizer.tokenize(words)
    words = [word for word in words if word not in stop_words]
    symptoms = []

    for w in words:
        w = ps.stem(w)
        symptoms.append(w)

    symptoms = "_".join(symptoms)
    all_symptoms_stemmed.append(symptoms)

In [5]:
import data_prep as data
from functools import reduce
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
tokenizer = RegexpTokenizer(r'\w+')

patterns, all_symptoms = data.load_data()

def load_stemmed_symptom_dict():
    
    stemmed_symptom_dict = {}
    for symptom in all_symptoms:

        words = symptom.replace("_", " ")
        words = tokenizer.tokenize(words)
        words = [w for w in words if w not in stop_words]

        stemmed_words = frozenset(ps.stem(w) for w in words)
        stemmed_symptom_dict[stemmed_words] = symptom
    
    return stemmed_symptom_dict

stemmed_symptom_dict = load_stemmed_symptom_dict()

def process_user_input(user_input):
    
    tokens = tokenizer.tokenize(user_input)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    
    stemmed_tokens = []
    for w in filtered_tokens:
        w = w.replace("_"," ")
        w = ps.stem(w)
        stemmed_tokens.append(w)
    
    return stemmed_tokens

def match_symptoms(stemmed_tokens):
    matched_symptoms = []
    for stemmed_key, original_symptom in stemmed_symptom_dict.items():
        if stemmed_key.issubset(stemmed_tokens):
            matched_symptoms.append(original_symptom)
    return matched_symptoms

def no_matched_symptoms(matched_symptoms):
    if len(matched_symptoms) == 0:
        print("\nSorry, I could not recognize any symptoms from what you described.")
        print("Please try to describe your symptoms more clearly.")
        print("Example: 'I have itching, skin rash and fever'")
        
def extract_symptoms(user_input):
    stemmed_tokens = process_user_input(user_input)
    matched_symptoms = match_symptoms(stemmed_tokens)
    no_matched_symptoms(matched_symptoms)
    print(matched_symptoms)
    
text = input("Describe your symptoms: ")
extract_symptoms(text)


['abdominal_pain', 'acidity', 'belly_pain']
